In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from statsforecast.models import (
    HistoricAverage,
    Naive,
    RandomWalkWithDrift,
    SeasonalNaive,
    SklearnModel,
)
from fpppy.utils import plot_series
from statsforecast import StatsForecast
from statsmodels.stats.diagnostic import acorr_ljungbox
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import rmse, mae, mape, mase
from sklearn.linear_model import LinearRegression
from functools import partial

In [ ]:
production_df = pd.read_csv("data/aus_production_formatted.csv",
  parse_dates=["ds"])
bricks_df = (
  production_df.loc[lambda x: (
    (x["unique_id"] == "Bricks") & (x["ds"].between("1970", "2004-12"))
  )].dropna()
)

In [ ]:
from fpppy.utils import plot_series
import matplotlib as mpl
import matplotlib.colors as mcolors

beers_df = production_df.loc[
    lambda x: (x["unique_id"] == "Beer") & (x["ds"] >= "1992")
]
train = beers_df.loc[lambda x: x["ds"] < "2007"]

avg_method = HistoricAverage()
naive_method = Naive()
seasonal_naive_method = SeasonalNaive(4)

sf = StatsForecast(
    models=[avg_method, naive_method, seasonal_naive_method],
    freq=pd.offsets.QuarterBegin(1),
)
sf.fit(train)

fcasts = sf.predict(h=14)

# utilsforecast.plot_series's `palette` relies on matplotlib colormaps.
# Some environments don't have `black_and_3color` pre-registered.
if "black_and_3color" not in mpl.colormaps:
    mpl.colormaps.register(
        mcolors.ListedColormap(["black", "#4C72B0", "#55A868", "#C44E52"]),
        name="black_and_3color",
    )

plot_series(
    train,
    fcasts,
    xlabel="Quarter",
    ylabel="Megalitres",
    title="Forecasts for quarterly beer production",
    palette="black_and_3color",
    rm_legend=False,
)

In [ ]:
import pandas as pd
goog_df = pd.read_csv('data/gafa_stock.csv', parse_dates=["ds"],sep=',', na_values=[''], quotechar='"').loc[lambda x:(x["unique_id"] == "GOOG_Close") & x["ds"].between("2015", "2016-01-31")]
train = goog_df.loc[lambda x: x["ds"].dt.year == 2015].copy()
test = goog_df.loc[lambda x: x["ds"].dt.year == 2016].copy()
print(test.head())
avg_method = HistoricAverage()
naive_method = Naive()
drift_method = RandomWalkWithDrift()
sf = StatsForecast(models=[avg_method, naive_method, drift_method],freq="1d")
sf.fit(train)
fcasts = sf.predict(h=len(test)).assign(y=test["y"].to_numpy())


plot_series(train, fcasts,)


In [ ]:
goog_df = (
  pd.read_csv("data/gafa_stock.csv", parse_dates=["ds"])
  .loc[lambda x: (
    (x["unique_id"] == "GOOG_Close") & x["ds"].between("2015", "2016-01-31")
  )]
)

train = goog_df.loc[lambda x: x["ds"].dt.year == 2015]
plot_series(train,
    xlabel="day [1]", ylabel="$US",
    title="Google daily closing stock prices in 2015")

In [ ]:
train

In [ ]:
resid = train.assign(resid=lambda x: x["y"].diff())
resid

In [ ]:
plot_series(resid,target_col="resid")

In [ ]:
ax = resid["resid"].hist(bins=20)
ax.set(title="Histogram of residuals", ylabel="count")
plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(resid["resid"][1:])

In [ ]:
fig = plot_acf(resid["resid"][1:])
#fig.axes[0].set(ylabel="acf")

In [ ]:
def plot_diagnostics(data):
    _, axes = plt.subplot_mosaic([["resid", "resid"], ["acf", "hist"]])
    ax = axes["resid"]
    ax.plot(data["ds"], data["resid"])
    ax.set(title="Innovation Residuals")
    ax = axes["acf"]
    plot_acf(data["resid"].dropna(),
        zero=False, bartlett_confint=False, auto_ylims=True, ax=ax)
    ax.set(title="ACF Plot", xlabel="lag[1]", ylabel="acf")
    ax = axes["hist"]
    ax.hist(data["resid"], bins=20)
    ax.set(title="Histogram", xlabel="resid", ylabel="count")

plt.rcParams["figure.figsize"] = (7, 5)
plot_diagnostics(resid)

In [ ]:
resid_test = acorr_ljungbox(resid["resid"].dropna(), boxpierce=True)
resid_test

In [ ]:
import pandas as pd
aus_production = pd.read_csv('data/aus_production_formatted.csv', sep=',', na_values=[''], quotechar='"')
aus_production

In [ ]:
aus_production.loc[aus_production["ds"] >= "1995"]
aus_production.loc[lambda x: x["ds"] >= "1995"]

In [ ]:
aus_production.query("ds >= '1995'")

In [ ]:
df = pd.DataFrame({"ds":[2000, 2005, 2010, 2015, 2020, 2025],"y":[1, 1, 2, 3, 5, 8]})
df

In [ ]:
df.iloc[2]

In [ ]:
aus_production.iloc[-20:]


In [ ]:
train_df = beers_df.loc[lambda x: x["ds"].dt.year <= 2007]
test_df = beers_df.loc[lambda x: x["ds"].dt.year > 2007]

mean_method = HistoricAverage()
naive_method = Naive()
drift_method = RandomWalkWithDrift()
seasonal_naive = SeasonalNaive(4)

sf = StatsForecast(freq="Q",
    models=[drift_method, mean_method, naive_method, seasonal_naive])
preds = (
    sf.forecast(df=train_df, h=10)
    .assign(y=test_df["y"].to_numpy())
)

plot_series(train_df, preds)

In [ ]:
evaluation = evaluate(preds, train_df=train_df,
    metrics=[rmse, mae, mape, partial(mase, seasonality=4)])

labels = {
    "method": "Method",
    "rmse": "RMSE", "mae": "MAE",
    "mape": "MAPE", "mase": "MASE",
}
evaluation

In [ ]:
evaluation_transformed = (
    evaluation
    .drop(columns=["unique_id"]).set_index("metric")
    .rename_axis("", axis="rows").rename_axis("Method", axis="columns")
    .transpose().reset_index().rename(columns=labels)
)
evaluation_transformed

In [ ]:
evaluation_transformed = (
    evaluation
    .drop(columns=["unique_id"]).set_index("metric")
    .rename_axis("Method", axis="columns")
    .transpose().reset_index().rename(columns=labels)
)
evaluation_transformed

In [ ]:
train_df = goog_df.loc[lambda x: x["ds"].dt.year == 2015].copy()
test_df = goog_df.loc[lambda x: x["ds"].dt.year == 2016].copy()
train_df["ds"] = np.arange(len(train_df["ds"]))
test_df["ds"] = len(train_df) + np.arange(len(test_df))

mean_method = HistoricAverage()
naive_method = Naive()
drift_method = RandomWalkWithDrift()
sf = StatsForecast(models=[drift_method, mean_method, naive_method], freq=1)
preds = (
    sf.forecast(df=train_df, h=len(test_df))
    .assign(y=test_df["y"].to_numpy())
)
preds

In [ ]:
evaluation = evaluate(preds, train_df=train_df,
    metrics=[rmse, mae, mape, partial(mase, seasonality=1)])
evaluation_transformed = (
    evaluation
    .drop(columns=["unique_id"]).set_index("metric")
    .rename_axis("Method", axis="columns")
    .transpose().reset_index().rename(columns=labels)
)
evaluation_transformed